# 📚 A/B Test Analysis & Interpretation: From Data to Decision
> Section 16 of the Statistics & A/B Testing curriculum  
> 통계 & A/B 테스트 커리큘럼 섹션 16

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Run the pre-registered statistical test on collected A/B data and compute the observed lift<br>사전등록된 통계 검정을 수집된 A/B 데이터에 실행하고 관측된 상승폭 계산하기
- [x] Distinguish statistical significance from practical significance using real numbers<br>실제 숫자로 통계적 유의성과 실질적 유의성 구분하기
- [x] Report a confidence interval on the lift itself, and check the effect's consistency across segments<br>상승폭 자체에 대한 신뢰구간을 보고하고, 세그먼트 전반에 걸친 효과의 일관성 확인하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** A/B test analysis is where the design from Section 14 and the sample size from Section 15 finally meet real data: running the pre-registered statistical test, computing the observed lift, checking whether it's both statistically AND practically significant, and quantifying uncertainty with a confidence interval on the lift itself — not just a yes/no significance verdict.

**KR:** A/B 테스트 분석은 섹션 14의 설계와 섹션 15의 표본 크기가 마침내 실제 데이터를 만나는 지점입니다: 사전등록된 통계 검정을 실행하고, 관측된 상승폭을 계산하며, 그것이 통계적으로도 실질적으로도 유의미한지 확인하고, 단순한 예/아니오 유의성 판정이 아니라 상승폭 자체에 대한 신뢰구간으로 불확실성을 정량화합니다.

### The analysis checklist (분석 체크리스트)

| Step / 단계 | Question it answers / 답하는 질문 | Reuses / 재사용 |
|---|---|---|
| Run the test / 검정 실행 | Is the observed lift likely real?<br>관측된 상승폭이 실제일 가능성이 있는가? | Section 7 |
| Practical significance / 실질적 유의성 | Is the effect big enough to matter?<br>효과가 신경 쓸 만큼 큰가? | Section 6 |
| CI on the lift / 상승폭의 신뢰구간 | How big is the effect, plausibly?<br>효과가 그럴듯하게 얼마나 큰가? | Section 4 |
| Segment analysis / 세그먼트 분석 | Does the effect hold up everywhere?<br>효과가 모든 곳에서 유지되는가? | Section 9 |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** A p-value alone answers "is this probably real?" but a business decision needs more: how big is the effect, is it big enough to matter, and does it hold up consistently across the different kinds of users who saw it? Skipping any of these steps risks either declaring a trivial win a "success" or missing a real one buried in noisy segment data.

**KR:** p-value 하나만으로는 "이것이 아마 실제일까?"에 답하지만, 비즈니스 결정에는 더 필요합니다: 효과가 얼마나 큰지, 신경 쓸 만큼 큰지, 그리고 그것을 본 서로 다른 종류의 사용자들에 걸쳐 일관되게 유지되는지. 이 단계 중 하나라도 건너뛰면 사소한 승리를 "성공"으로 선언하거나, 노이즈 많은 세그먼트 데이터에 묻힌 진짜 효과를 놓칠 위험이 있습니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** The moment a pre-registered test finishes collecting its planned sample — turning raw event logs into the specific numbers (lift, p-value, CI, segment breakdown) that go into a launch decision memo.

**KR:** 사전등록된 테스트가 계획된 표본 수집을 마치는 순간 -- 원본 이벤트 로그를 출시 결정 메모에 들어갈 구체적인 숫자(상승폭, p-value, 신뢰구간, 세그먼트 분석)로 바꿉니다.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [9]:
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Collected A/B test data: conversion counts per group
# 수집된 A/B 테스트 데이터: 그룹별 전환 횟수
np.random.seed(42)
n_control, n_treatment = 5000, 5000
control_conversions = np.random.binomial(1, 0.10, n_control)
treatment_conversions = np.random.binomial(1, 0.115, n_treatment)

control_rate = control_conversions.mean()
treatment_rate = treatment_conversions.mean()
lift = treatment_rate - control_rate

# Two-proportion z-test
count = np.array([treatment_conversions.sum(), control_conversions.sum()])
nobs = np.array([n_treatment, n_control])
z_stat, p_value = proportions_ztest(count, nobs)

print(f"Control: {control_rate:.2%}, Treatment: {treatment_rate:.2%}")
print(f"Absolute lift: {lift:+.2%}, Relative lift: {lift / control_rate:+.1%}")
print(f"z-statistic: {z_stat:.3f}, p-value: {p_value:.4f}")

Control: 9.58%, Treatment: 10.90%
Absolute lift: +1.32%, Relative lift: +13.8%
z-statistic: 2.177, p-value: 0.0295


## Common Variations (자주 쓰는 변형)

In [10]:
# One-sided test if the hypothesis is specifically "treatment is BETTER"
# 가설이 특히 "실험군이 더 낫다"인 경우의 단측검정
z_stat_one, p_value_one = proportions_ztest(count, nobs, alternative='larger')
print(f"One-sided p-value (treatment > control specifically): {p_value_one:.4f}")

One-sided p-value (treatment > control specifically): 0.0147


---
# 🧪 Small Examples (예제)

## Example 1 — Running the Analysis: From Raw Data to Lift and p-value (원본 데이터에서 상승폭과 p-value까지)

In [11]:
# Business question: the pre-registered test has finished collecting data.
# What's the actual lift, and is it statistically significant?
# 비즈니스 질문: 사전등록된 테스트가 데이터 수집을 마쳤습니다. 실제
# 상승폭은 얼마이며, 통계적으로 유의미한가?

np.random.seed(1)
n_control, n_treatment = 6200, 6180
control_conv = np.random.binomial(1, 0.084, n_control)
treatment_conv = np.random.binomial(1, 0.098, n_treatment)

control_rate = control_conv.mean()
treatment_rate = treatment_conv.mean()
absolute_lift = treatment_rate - control_rate
relative_lift = absolute_lift / control_rate

count = np.array([treatment_conv.sum(), control_conv.sum()])
nobs = np.array([n_treatment, n_control])
z_stat, p_value = proportions_ztest(count, nobs)

print(f"Control: {control_rate:.2%} (n={n_control})")
print(f"Treatment: {treatment_rate:.2%} (n={n_treatment})")
print(f"Absolute lift: {absolute_lift:+.2%}, Relative lift: {relative_lift:+.1%}")
print(f"z-statistic: {z_stat:.3f}, p-value: {p_value:.4f}")
print()
alpha = 0.05
if p_value < alpha:
    print(f"p={p_value:.4f} < {alpha} -> statistically significant")
    print(f"p={p_value:.4f} < {alpha} -> 통계적으로 유의미함")
else:
    print(f"p={p_value:.4f} >= {alpha} -> not statistically significant")
    print(f"p={p_value:.4f} >= {alpha} -> 통계적으로 유의미하지 않음")

Control: 8.18% (n=6200)
Treatment: 9.40% (n=6180)
Absolute lift: +1.22%, Relative lift: +15.0%
z-statistic: 2.405, p-value: 0.0162

p=0.0162 < 0.05 -> statistically significant
p=0.0162 < 0.05 -> 통계적으로 유의미함


## Example 2 — Statistical vs Practical Significance, With Real Numbers (실제 숫자로 보는 통계적 vs 실질적 유의성)

In [12]:
# Business question: Test A found a "significant" 0.1pp lift from 1,000,000
# users per group. Test B found a "not yet significant" 2pp lift from 400
# users per group. Which result is more actionable for the business?
# 비즈니스 질문: 테스트 A는 그룹당 100만 명에서 "유의미한" 0.1%p 상승을
# 발견했습니다. 테스트 B는 그룹당 400명에서 "아직 유의미하지 않은" 2%p
# 상승을 발견했습니다. 어느 결과가 비즈니스적으로 더 실행 가능한가?

np.random.seed(2)

# Test A: huge sample, tiny lift
n_a = 1000000
control_a = np.random.binomial(1, 0.100, n_a)
treatment_a = np.random.binomial(1, 0.101, n_a)
count_a = np.array([treatment_a.sum(), control_a.sum()])
nobs_a = np.array([n_a, n_a])
z_a, p_a = proportions_ztest(count_a, nobs_a)
lift_a = treatment_a.mean() - control_a.mean()

# Test B: small sample, bigger lift
n_b = 400
control_b = np.random.binomial(1, 0.100, n_b)
treatment_b = np.random.binomial(1, 0.120, n_b)
count_b = np.array([treatment_b.sum(), control_b.sum()])
nobs_b = np.array([n_b, n_b])
z_b, p_b = proportions_ztest(count_b, nobs_b)
lift_b = treatment_b.mean() - control_b.mean()

print(f"Test A (n={n_a:,} per group): lift={lift_a:+.3%}, p={p_a:.4f}")
print(f"Test B (n={n_b} per group):     lift={lift_b:+.3%}, p={p_b:.4f}")
print()
print("Test A is statistically significant but the lift is tiny -- likely not worth")
print("engineering effort to ship. Test B shows a much bigger lift but the small")
print("sample means the evidence isn't yet conclusive -- worth a bigger follow-up test,")
print("not a dismissal. 'Significant' and 'big enough to matter' are different questions.")
print("테스트 A는 통계적으로 유의미하지만 상승폭이 작아서 출시할 만한 엔지니어링")
print("비용의 가치가 없을 가능성이 큽니다. 테스트 B는 훨씬 큰 상승폭을 보이지만")
print("작은 표본 때문에 증거가 아직 결정적이지 않습니다 -- 기각이 아니라 더 큰")
print("후속 테스트가 필요합니다. '유의미함'과 '신경 쓸 만큼 큼'은 다른 질문입니다.")

Test A (n=1,000,000 per group): lift=+0.145%, p=0.0007
Test B (n=400 per group):     lift=+2.000%, p=0.4041

Test A is statistically significant but the lift is tiny -- likely not worth
engineering effort to ship. Test B shows a much bigger lift but the small
sample means the evidence isn't yet conclusive -- worth a bigger follow-up test,
not a dismissal. 'Significant' and 'big enough to matter' are different questions.
테스트 A는 통계적으로 유의미하지만 상승폭이 작아서 출시할 만한 엔지니어링
비용의 가치가 없을 가능성이 큽니다. 테스트 B는 훨씬 큰 상승폭을 보이지만
작은 표본 때문에 증거가 아직 결정적이지 않습니다 -- 기각이 아니라 더 큰
후속 테스트가 필요합니다. '유의미함'과 '신경 쓸 만큼 큼'은 다른 질문입니다.


## Example 3 — Confidence Interval on the Lift Itself (상승폭 자체의 신뢰구간)

In [13]:
# Business question: instead of just "the lift was 1.4 percentage points,"
# give leadership an honest range for what the true lift plausibly is.
# 비즈니스 질문: "상승폭이 1.4%p였다"뿐 아니라, 실제 상승폭이 그럴듯하게
# 어느 범위에 있는지 경영진에게 정직하게 제시하기.

np.random.seed(3)
n_control, n_treatment = 8000, 8000
control_conv = np.random.binomial(1, 0.09, n_control)
treatment_conv = np.random.binomial(1, 0.104, n_treatment)

p_control = control_conv.mean()
p_treatment = treatment_conv.mean()
lift = p_treatment - p_control

# Standard error of the DIFFERENCE between two proportions (Section 4's approach, extended)
se_diff = np.sqrt(p_control * (1 - p_control) / n_control + p_treatment * (1 - p_treatment) / n_treatment)
z_critical = stats.norm.ppf(0.975)  # 95% CI
ci_lower = lift - z_critical * se_diff
ci_upper = lift + z_critical * se_diff

print(f"Observed lift: {lift:+.2%}")
print(f"95% CI for the lift: [{ci_lower:+.2%}, {ci_upper:+.2%}]")
print()
if ci_lower > 0:
    print("The entire CI is above 0 -- we can be reasonably confident the true lift is")
    print("positive, and the range tells leadership the plausible best/worst case.")
    print("신뢰구간 전체가 0보다 위입니다 -- 실제 상승폭이 양수라고 상당히 확신할 수")
    print("있으며, 이 범위는 경영진에게 그럴듯한 최선·최악의 경우를 알려줍니다.")
else:
    print("The CI crosses 0 -- we cannot be confident the true effect is even positive.")
    print("신뢰구간이 0을 걸쳐 있습니다 -- 실제 효과가 양수인지조차 확신할 수 없습니다.")

Observed lift: +1.59%
95% CI for the lift: [+0.67%, +2.51%]

The entire CI is above 0 -- we can be reasonably confident the true lift is
positive, and the range tells leadership the plausible best/worst case.
신뢰구간 전체가 0보다 위입니다 -- 실제 상승폭이 양수라고 상당히 확신할 수
있으며, 이 범위는 경영진에게 그럴듯한 최선·최악의 경우를 알려줍니다.


## Example 4 — Segment Analysis: Does the Effect Hold Up Everywhere? (효과가 모든 곳에서 유지되는가?)

In [14]:
# Business question: does the treatment effect hold up consistently for both
# new and returning users, or is the overall result driven by just one segment?
# 비즈니스 질문: 처치 효과가 신규·기존 사용자 모두에서 일관되게 유지되는가,
# 아니면 전체 결과가 단 하나의 세그먼트에서만 나온 것인가?

np.random.seed(4)
n_per_segment = 2500

segments = {}
for segment, true_lift in [('new_users', 0.015), ('returning_users', 0.014)]:
    control = np.random.binomial(1, 0.09, n_per_segment)
    treatment = np.random.binomial(1, 0.09 + true_lift, n_per_segment)
    count = np.array([treatment.sum(), control.sum()])
    nobs = np.array([n_per_segment, n_per_segment])
    z, p = proportions_ztest(count, nobs)
    segments[segment] = {'control_rate': control.mean(), 'treatment_rate': treatment.mean(),
                          'lift': treatment.mean() - control.mean(), 'p_value': p}

for segment, result in segments.items():
    print(f"{segment}: control={result['control_rate']:.2%}, treatment={result['treatment_rate']:.2%}, "
          f"lift={result['lift']:+.2%}, p={result['p_value']:.4f}")

print()
print("The effect is consistent across both segments -- reassuring. If instead ONE")
print("segment out of many showed 'significant' and the rest didn't, that pattern alone")
print("(checking many segments = many tests) reintroduces Section 9's multiple-comparisons")
print("problem -- a 'significant' result in 1 of 10 segments checked isn't a real finding,")
print("it's close to what you'd expect by chance alone.")
print("효과가 두 세그먼트 모두에서 일관됩니다 -- 안심되는 결과입니다. 만약 대신")
print("여러 세그먼트 중 하나만 '유의미'하고 나머지는 아니었다면, 그 패턴 자체가")
print("(여러 세그먼트 확인 = 여러 번의 검정) 섹션 9의 다중비교 문제를 다시")
print("불러옵니다 -- 확인한 10개 세그먼트 중 1개의 '유의미한' 결과는 진짜 발견이")
print("아니라, 순전히 우연으로 예상되는 것에 가깝습니다.")

new_users: control=9.76%, treatment=10.96%, lift=+1.20%, p=0.1639
returning_users: control=9.04%, treatment=10.44%, lift=+1.40%, p=0.0950

The effect is consistent across both segments -- reassuring. If instead ONE
segment out of many showed 'significant' and the rest didn't, that pattern alone
(checking many segments = many tests) reintroduces Section 9's multiple-comparisons
problem -- a 'significant' result in 1 of 10 segments checked isn't a real finding,
it's close to what you'd expect by chance alone.
효과가 두 세그먼트 모두에서 일관됩니다 -- 안심되는 결과입니다. 만약 대신
여러 세그먼트 중 하나만 '유의미'하고 나머지는 아니었다면, 그 패턴 자체가
(여러 세그먼트 확인 = 여러 번의 검정) 섹션 9의 다중비교 문제를 다시
불러옵니다 -- 확인한 10개 세그먼트 중 1개의 '유의미한' 결과는 진짜 발견이
아니라, 순전히 우연으로 예상되는 것에 가깝습니다.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [15]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: run the analysis -- a completed test's data ----------
np.random.seed(5)
n_control_a, n_treatment_a = 3000, 3000
control_a = np.random.binomial(1, 0.06, n_control_a)
treatment_a = np.random.binomial(1, 0.072, n_treatment_a)

alpha_a = 0.05   # TODO: significance level to compare the p-value against, e.g. 0.05

if ready(alpha_a):
    count_a = np.array([treatment_a.sum(), control_a.sum()])
    nobs_a = np.array([n_treatment_a, n_control_a])
    z_a, p_a = proportions_ztest(count_a, nobs_a)
    print(f"Control: {control_a.mean():.2%}, Treatment: {treatment_a.mean():.2%}")
    print(f"z={z_a:.3f}, p={p_a:.4f}")
    tag = "Significant" if p_a < alpha_a else "Not significant"
    print(f"{tag} at alpha={alpha_a}")
else:
    print("Part A \u270f\ufe0f  Fill in alpha_a above, then re-run.")

# ---------- Part B: statistical vs practical significance ----------
# Scenario: n=500,000 per group found p=0.001 but the lift is only 0.05 percentage points.
lift_b = 0.0005
p_value_b = 0.001

practically_significant_b = True   # TODO: True or False -- would you recommend shipping
                                    # this based on the lift size alone (assume shipping
                                    # costs meaningful engineering effort)?

if ready(practically_significant_b):
    print(f"Lift: {lift_b:.3%}, p-value: {p_value_b}")
    print(f"Your judgment: practically significant = {practically_significant_b}")
    print("Statistical significance and practical significance are different questions --")
    print("a p=0.001 result with a 0.05pp lift is a real but likely tiny effect.")
else:
    print("Part B \u270f\ufe0f  Fill in practically_significant_b above (True or False), then re-run.")

# ---------- Part C: CI on the lift -- delivery time improvement ----------
np.random.seed(6)
n_c = 1500
control_c = np.random.normal(45, 12, n_c)  # minutes
treatment_c = np.random.normal(41, 12, n_c)

confidence_c = 0.95   # TODO: confidence level, e.g. 0.95

if ready(confidence_c):
    lift_c = treatment_c.mean() - control_c.mean()
    se_c = np.sqrt(control_c.std(ddof=1) ** 2 / n_c + treatment_c.std(ddof=1) ** 2 / n_c)
    z_c = stats.norm.ppf(1 - (1 - confidence_c) / 2)
    ci_lower_c = lift_c - z_c * se_c
    ci_upper_c = lift_c + z_c * se_c
    print(f"Lift: {lift_c:+.2f} min, {confidence_c:.0%} CI: [{ci_lower_c:+.2f}, {ci_upper_c:+.2f}]")
else:
    print("Part C \u270f\ufe0f  Fill in confidence_c above, then re-run.")

# ---------- Part D: segment analysis -- how many "tests" are really happening? ----------
segments_checked_d = ['new_users', 'returning_users', 'mobile', 'desktop', 'US', 'non-US']

n_segments_d = 6   # TODO: how many segments are being checked here? (a number)

if ready(n_segments_d):
    false_positive_chance = 1 - (1 - 0.05) ** n_segments_d
    print(f"Checking {n_segments_d} segments separately, each at alpha=0.05:")
    print(f"chance of at least one false 'significant' segment purely by luck = {false_positive_chance:.1%}")
    print("This is exactly Section 9's multiple-comparisons problem, applied to segments.")
else:
    print("Part D \u270f\ufe0f  Fill in n_segments_d above, then re-run.")

Control: 5.97%, Treatment: 7.90%
z=2.948, p=0.0032
Significant at alpha=0.05
Lift: 0.050%, p-value: 0.001
Your judgment: practically significant = True
Statistical significance and practical significance are different questions --
a p=0.001 result with a 0.05pp lift is a real but likely tiny effect.
Lift: -3.30 min, 95% CI: [-4.13, -2.47]
Checking 6 segments separately, each at alpha=0.05:
chance of at least one false 'significant' segment purely by luck = 26.5%
This is exactly Section 9's multiple-comparisons problem, applied to segments.


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Running a Different Test Than What Was Pre-Registered (사전등록과 다른 검정 실행)
If Section 14's design specified a two-sided test but the analysis quietly switches to one-sided (which halves the p-value and makes significance easier to reach), this is a direct violation of the pre-registration — even if it "feels" justified after seeing the direction of the result.  
섹션 14의 설계가 양측검정을 명시했는데 분석에서 조용히 단측검정으로 바꾼다면(p-value가 절반이 되어 유의성 도달이 더 쉬워짐), 결과의 방향을 본 뒤 "타당해 보인다"고 느껴져도 이는 사전등록의 직접적인 위반입니다.

✅ **Fix / 해결:** Run exactly the test specified in the pre-registered design document, decided before the direction of the result was known.

### Mistake 2 — Treating "Statistically Significant" as Automatically "Ship It" (통계적으로 유의미하면 자동으로 "출시"라고 취급)
As Example 2 showed, a tiny lift can be statistically significant with a large enough sample — "p<0.05" answers "is this probably real," not "is this worth the engineering cost of shipping."  
예제 2가 보여줬듯, 표본이 충분히 크면 작은 상승폭도 통계적으로 유의미할 수 있습니다 -- "p<0.05"는 "이것이 아마 실제인가"에 답하지, "출시할 만한 엔지니어링 비용의 가치가 있는가"에 답하지 않습니다.

✅ **Fix / 해결:** Always check the lift size against a business-meaningful threshold (ideally the MDE from Section 15) before recommending a launch.

### Mistake 3 — Reporting a Lift Without Its Confidence Interval (신뢰구간 없이 상승폭 보고)
"The lift was +1.4 percentage points" sounds precise but hides how much that number could plausibly vary — the same headline number with a CI of [+1.3%, +1.5%] and one with [-0.2%, +3.0%] tell very different stories.  
"상승폭이 +1.4%p였다"는 정밀해 보이지만 그 숫자가 그럴듯하게 얼마나 달라질 수 있는지를 숨깁니다 -- 같은 헤드라인 숫자라도 신뢰구간이 [+1.3%, +1.5%]인 경우와 [-0.2%, +3.0%]인 경우는 매우 다른 이야기를 합니다.

✅ **Fix / 해결:** Always report the lift alongside its confidence interval (Example 3's approach), not as a single bare number.

### Mistake 4 — Checking Many Segments and Reporting Only the "Significant" One (여러 세그먼트를 확인하고 "유의미한" 것만 보고)
Checking 10 segments and highlighting the 1 that happened to hit p<0.05 while ignoring the other 9 is p-hacking wearing a "deep dive" costume — Example 4's multiple-comparisons math applies just as much to segments as it does to metrics.  
10개의 세그먼트를 확인하고 우연히 p<0.05를 넘긴 1개만 부각하며 나머지 9개는 무시하는 것은 "심층 분석"이라는 옷을 입은 p-해킹입니다 -- 예제 4의 다중비교 수학은 지표만큼이나 세그먼트에도 그대로 적용됩니다.

✅ **Fix / 해결:** Pre-specify which segments matter (as part of Section 14's design) and apply a multiple-comparisons correction if checking several, rather than fishing for a significant one after the fact.

### Mistake 5 — Analyzing Before the Pre-Registered Sample Size Is Reached (사전등록된 표본 크기에 도달하기 전에 분석)
Peeking at the analysis before Section 15's calculated sample size has actually been collected is Section 14's peeking problem wearing an "analysis" costume instead of a "design" one — the false-positive inflation is identical.  
섹션 15에서 계산한 표본 크기가 실제로 수집되기 전에 분석을 엿보는 것은 섹션 14의 엿보기 문제가 "설계" 대신 "분석"이라는 옷을 입은 것입니다 -- 오탐 부풀림은 동일합니다.

✅ **Fix / 해결:** Wait until the pre-registered sample size (or duration) is actually reached before running the final analysis that will inform a decision.

### Mistake 6 — Treating a "Not Significant" Segment as Proof That Segment Has No Effect (유의하지 않은 세그먼트를 효과 없음의 증거로 취급)
Splitting an already-sized sample into segments cuts the sample size (and thus statistical power) within each one — a segment showing "not significant" might just be underpowered on its own, not evidence the effect genuinely differs there (Section 6's Mistake 4, applied per-segment).  
이미 크기가 정해진 표본을 세그먼트로 나누면 각 세그먼트 내의 표본 크기(그리고 검정력)가 줄어듭니다 -- "유의하지 않은" 세그먼트는 그저 그 자체로 검정력이 부족한 것일 수 있지, 그곳에서 효과가 실제로 다르다는 증거가 아닙니다(섹션 6의 실수 4를 세그먼트별로 적용한 것).

✅ **Fix / 해결:** Interpret segment-level "non-significance" cautiously, checking whether the segment's sample size alone provides adequate power before concluding the effect differs there.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- Compute the observed lift and its direction FIRST, before running any significance test — it's easy to accidentally flip a sign (treatment minus control vs control minus treatment) and misreport which group actually won.<br>어느 검정을 돌리기 전에 관측된 상승폭과 방향을 먼저 계산하세요 -- 부호를 실수로 뒤집기 쉽고(실험군-대조군 vs 대조군-실험군) 어느 그룹이 실제로 이겼는지 잘못 보고할 수 있습니다.
- `proportions_ztest`'s `count` and `nobs` arguments both take arrays in the SAME order (e.g., [treatment, control]) — mixing up the order for one but not the other silently corrupts the result.<br>proportions_ztest의 count와 nobs 인자는 둘 다 같은 순서(예: [실험군, 대조군])로 배열을 받습니다 -- 하나만 순서를 바꾸면 조용히 결과가 망가집니다.
- A confidence interval that excludes 0 is the direct visual equivalent of "p < alpha" for a two-sided test — if you can see the CI, you often don't even need to separately state the p-value.<br>0을 포함하지 않는 신뢰구간은 양측검정에서 "p < alpha"의 직접적인 시각적 대응물입니다 -- 신뢰구간을 볼 수 있다면 종종 p-value를 따로 말할 필요조차 없습니다.
- When segment results look inconsistent, resist jumping straight to "the effect is different for this segment" — first check whether each segment simply has a much smaller sample (and thus wider, overlapping confidence intervals) than the overall test.<br>세그먼트 결과가 일관되지 않아 보일 때, 바로 "이 세그먼트는 효과가 다르다"고 넘어가지 마세요 -- 먼저 각 세그먼트가 전체 테스트보다 훨씬 작은 표본(그래서 더 넓고 겹치는 신뢰구간)을 갖고 있는 건 아닌지 확인하세요.
- Write the analysis summary in the same structure every time (lift, CI, p-value, segment check) — a consistent template makes it much harder to accidentally omit the step that would have caught a problem.<br>분석 요약은 매번 같은 구조(상승폭, 신뢰구간, p-value, 세그먼트 확인)로 작성하세요 -- 일관된 템플릿은 문제를 잡아냈을 단계를 실수로 빠뜨리기 훨씬 어렵게 만듭니다.

---
# 🔗 Related Concepts (관련 개념)

```
Sample Size & Power Analysis (Section 15)
   MDE, sample size, power curves, sample size -> duration
        ↓
A/B Test Analysis & Interpretation (Section 16)  ← you are here / 현재 위치
   running the test, statistical vs practical significance, CI on the lift, segments
        ↓
A/B Testing Pitfalls (Section 17)
   peeking, novelty effects, Simpson's paradox, SRM (deeper dive)
```

*How is today's topic connected to other concepts?*

**EN:** This section is where nearly every prior section gets used at once on real numbers: the test itself is Section 7's proportion/mean test, the CI on the lift is Section 4's machinery extended to a difference, and the segment-analysis warning is Section 9's multiple-comparisons problem showing up yet again in a new context. Section 17 goes deeper into exactly the traps this section only touched on — what happens when a test IS peeked at early, when an effect fades after the "novelty" of a change wears off, and when an aggregate result reverses once you look inside it (Simpson's paradox).

**KR:** 이 섹션은 거의 모든 이전 섹션이 실제 숫자에 대해 한 번에 사용되는 지점입니다: 검정 자체는 섹션 7의 비율·평균 검정이고, 상승폭의 신뢰구간은 섹션 4의 도구를 차이로 확장한 것이며, 세그먼트 분석 경고는 섹션 9의 다중비교 문제가 새로운 맥락에서 또다시 등장한 것입니다. 섹션 17은 이 섹션이 살짝만 다뤘던 함정들을 더 깊이 파고듭니다 -- 테스트를 실제로 일찍 엿봤을 때, 변경의 "새로움"이 사라진 뒤 효과가 희미해질 때, 그리고 전체 결과가 안을 들여다보면 뒤집힐 때(심슨의 역설) 무슨 일이 일어나는지.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** The pre-registered checkout redesign test just reached its planned sample size. Write the complete analysis: the lift, whether it's statistically AND practically significant, the confidence interval, and whether the effect holds up across new vs returning customers — everything a launch decision memo needs.

**KR:** 사전등록된 결제 재설계 테스트가 방금 계획된 표본 크기에 도달했습니다. 완전한 분석을 작성합니다: 상승폭, 통계적으로도 실질적으로도 유의미한지, 신뢰구간, 그리고 신규·기존 고객 모두에서 효과가 유지되는지 -- 출시 결정 메모에 필요한 모든 것.

**To-do / 할 일:**
- [x] Compute the lift, p-value, and confidence interval<br>상승폭, p-value, 신뢰구간 계산하기
- [x] Check practical significance against the pre-registered MDE<br>사전등록된 MDE와 비교해 실질적 유의성 확인하기
- [x] Check consistency across new vs returning customer segments<br>신규·기존 고객 세그먼트 간 일관성 확인하기
- [x] Write a launch recommendation<br>출시 제안 작성하기

In [16]:
np.random.seed(80)
n_control, n_treatment = 8000, 8000
control_conv = np.random.binomial(1, 0.072, n_control)
treatment_conv = np.random.binomial(1, 0.085, n_treatment)

p_control = control_conv.mean()
p_treatment = treatment_conv.mean()
lift = p_treatment - p_control
relative_lift = lift / p_control

count = np.array([treatment_conv.sum(), control_conv.sum()])
nobs = np.array([n_treatment, n_control])
z_stat, p_value = proportions_ztest(count, nobs)

se_diff = np.sqrt(p_control * (1 - p_control) / n_control + p_treatment * (1 - p_treatment) / n_treatment)
z_crit = stats.norm.ppf(0.975)
ci_lower, ci_upper = lift - z_crit * se_diff, lift + z_crit * se_diff

mde_from_design = 0.005  # this test's pre-registered MDE (Section 15)

print("=== A/B Test Analysis: Checkout Redesign ===")
print(f"Control: {p_control:.2%} (n={n_control}), Treatment: {p_treatment:.2%} (n={n_treatment})")
print(f"Absolute lift: {lift:+.2%} (relative: {relative_lift:+.1%})")
print(f"p-value: {p_value:.4f}")
print(f"95% CI for the lift: [{ci_lower:+.2%}, {ci_upper:+.2%}]")
print()

statistically_sig = p_value < 0.05
practically_sig = lift >= mde_from_design
print(f"Statistically significant: {statistically_sig}")
print(f"Practically significant (lift >= pre-registered MDE of {mde_from_design:.1%}): {practically_sig}")
print()

# Segment check
for segment in ['new_customers', 'returning_customers']:
    n_seg = n_control // 2
    c_seg = np.random.binomial(1, p_control, n_seg)
    t_seg = np.random.binomial(1, p_treatment, n_seg)
    seg_lift = t_seg.mean() - c_seg.mean()
    count_seg = np.array([t_seg.sum(), c_seg.sum()])
    nobs_seg = np.array([n_seg, n_seg])
    _, p_seg = proportions_ztest(count_seg, nobs_seg)
    print(f"{segment}: lift={seg_lift:+.2%}, p={p_seg:.4f}")

print()
if statistically_sig and practically_sig:
    print("Recommendation: LAUNCH. The lift is both statistically significant and")
    print("exceeds the pre-registered minimum worth caring about, with a CI that")
    print("stays positive throughout.")
    print()
    print("제안: 출시. 상승폭이 통계적으로 유의미하고 사전등록된 최소 기준을 넘으며,")
    print("신뢰구간도 전체적으로 양수를 유지합니다.")
else:
    print("Recommendation: hold off -- either statistical or practical significance")
    print("criteria were not met. Review the numbers above before deciding next steps.")

=== A/B Test Analysis: Checkout Redesign ===
Control: 7.14% (n=8000), Treatment: 8.04% (n=8000)
Absolute lift: +0.90% (relative: +12.6%)
p-value: 0.0316
95% CI for the lift: [+0.08%, +1.72%]

Statistically significant: True
Practically significant (lift >= pre-registered MDE of 0.5%): True

new_customers: lift=+0.23%, p=0.6987
returning_customers: lift=+1.50%, p=0.0116

Recommendation: LAUNCH. The lift is both statistically significant and
exceeds the pre-registered minimum worth caring about, with a CI that
stays positive throughout.

제안: 출시. 상승폭이 통계적으로 유의미하고 사전등록된 최소 기준을 넘으며,
신뢰구간도 전체적으로 양수를 유지합니다.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** A/B test analysis turns collected data into a launch decision by running the pre-registered test (Section 7's proportion/mean tests) to get an observed lift and p-value, checking that lift against a business-meaningful threshold (practical significance, not just statistical), reporting a confidence interval on the lift itself rather than a single bare number (Section 4's machinery, extended to a difference), and verifying the effect holds up reasonably consistently across key user segments. The segment-checking step carries its own risk: examining many segments and highlighting only the one that happens to reach significance reintroduces Section 9's multiple-comparisons problem in a new disguise, and a "not significant" segment result often just reflects a smaller, underpowered sample rather than proof the effect genuinely differs there. Every piece of this analysis reuses a tool from earlier in the curriculum — this section is where they finally get applied together, on real numbers, to make an actual business call.

**KR:** A/B 테스트 분석은 사전등록된 검정(섹션 7의 비율·평균 검정)을 실행해 관측된 상승폭과 p-value를 얻고, 그 상승폭을 비즈니스적으로 의미 있는 기준(단순한 통계적 유의성이 아니라 실질적 유의성)과 비교하며, 단일 숫자가 아니라 상승폭 자체에 대한 신뢰구간(섹션 4의 도구를 차이로 확장)을 보고하고, 효과가 주요 사용자 세그먼트 전반에 걸쳐 합리적으로 일관되게 유지되는지 검증함으로써 수집된 데이터를 출시 결정으로 바꿉니다. 세그먼트 확인 단계는 자체적인 위험을 안고 있습니다: 여러 세그먼트를 살펴보고 우연히 유의성에 도달한 것만 부각하면 섹션 9의 다중비교 문제가 새로운 형태로 다시 등장하며, "유의하지 않은" 세그먼트 결과는 종종 그저 더 작고 검정력이 부족한 표본을 반영할 뿐, 그곳에서 효과가 정말로 다르다는 증거가 아닙니다. 이 분석의 모든 부분은 커리큘럼 앞부분의 도구를 재사용합니다 -- 이 섹션은 그것들이 마침내 실제 숫자에 함께 적용되어 실제 비즈니스 결정을 내리는 지점입니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** A/B test analysis runs the pre-registered test to get a lift and p-value, checks that lift against a business-meaningful threshold (not just statistical significance), reports a confidence interval on the lift instead of a bare number, and checks segment consistency carefully — since fishing through many segments for one "significant" result reintroduces the multiple-comparisons trap in disguise.

> **KR:** A/B 테스트 분석은 사전등록된 검정을 실행해 상승폭과 p-value를 얻고, 그 상승폭을 비즈니스적으로 의미 있는 기준(단순한 통계적 유의성이 아니라)과 비교하며, 단일 숫자 대신 상승폭에 대한 신뢰구간을 보고하고, 세그먼트 일관성을 신중히 확인합니다 -- 여러 세그먼트를 뒤져 유의미한 결과 하나를 찾는 것은 다중비교 함정을 변장시켜 다시 끌어들이는 것이기 때문입니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** Why is switching from a pre-registered two-sided test to a one-sided test AFTER seeing the direction of the result a violation of good practice?  
**Q1 (KR).** 결과의 방향을 본 뒤 사전등록된 양측검정에서 단측검정으로 바꾸는 것이 왜 좋은 관행의 위반인가요?

Switching to one-sided after seeing the direction halves the p-value, so a weak result looks significant.  
방향을 본 뒤 단측으로 바꾸면 p-value가 절반이 되어 약한 결과가 유의해 보입니다.

**Q2 (EN).** A test shows p=0.002 but the lift is only 0.03 percentage points. Should this automatically be shipped? What else needs to be checked?  
**Q2 (KR).** 어떤 테스트가 p=0.002를 보여주지만 상승폭은 0.03%p에 불과합니다. 이것이 자동으로 출시되어야 하나요? 그 외에 무엇을 확인해야 하나요?

No; check whether a 0.03 point lift is worth the cost, plus the confidence interval and guardrail metrics.  
아니요. 0.03%p 상승이 비용만큼 가치가 있는지, 신뢰구간과 가드레일 지표를 확인해야 합니다.

**Q3 (EN).** Why does reporting "the lift was +1.4%" without a confidence interval hide important information?  
**Q3 (KR).** 신뢰구간 없이 "상승폭이 +1.4%였다"고 보고하는 것이 왜 중요한 정보를 숨기나요?

A single number hides the uncertainty; the interval shows the range of plausible lifts, which may include zero.  
숫자 하나는 불확실성을 숨깁니다. 신뢰구간은 가능한 상승폭의 범위를 보여주며 0을 포함할 수도 있습니다.

**Q4 (EN).** A team checks 8 user segments and finds that exactly 1 shows p<0.05. Why is this alone not strong evidence the effect is real in that segment?  
**Q4 (KR).** 팀이 사용자 세그먼트 8개를 확인했더니 정확히 1개에서 p<0.05가 나왔습니다. 이것만으로 왜 그 세그먼트에서 효과가 실제라는 강한 증거가 되지 않나요?

Checking 8 segments gives about a 34% chance of at least one false positive, so one hit can be luck.  
세그먼트 8개를 보면 적어도 하나가 거짓 양성일 확률이 약 34%라서, 1개는 우연일 수 있습니다.

**Q5 (EN).** A specific segment shows "not significant" results. Why doesn't this necessarily mean the treatment has no effect for that segment?  
**Q5 (KR).** 특정 세그먼트가 "유의하지 않은" 결과를 보입니다. 이것이 왜 그 세그먼트에서 처치가 효과가 없다는 것을 반드시 의미하지는 않나요?

"Not significant" means not enough evidence, often from a small sample, not zero effect.  
"유의하지 않음"은 증거 부족이라는 뜻이고, 표본이 작아서 생기는 경우가 많으며 효과가 0이라는 뜻이 아닙니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*